In [6]:
from hgp  import LDPCCodeGenerator
import pickle 
from ldpc.code_util import estimate_code_distance, search_cycles
  # update filename as needed

In [ ]:
generator = LDPCCodeGenerator()

for n in range(4,21):

    output_file_name = f"_candidate_code_{n}.pkl"
    num_logical_bits = n
    num_bits = int(4 * num_logical_bits)
    num_checks = int(3 * num_logical_bits)
    num_checks_per_col = 3
    code_distance = 6

    generator.generate_until_valid(
        output_file_name,
        num_checks,
        num_bits,
        num_checks_per_col,
        code_distance,
        code_type="regular",
        construction_method="peg"
    )

In [7]:
generator = LDPCCodeGenerator()

for n in range(4, 21):
    best_matrix = None
    best_distance = 0
    best_seed = None
    best_meta = None

    num_bits = int(4 * n)
    num_checks = int(3 * n)
    num_checks_per_col = 3

    print(f"\n=== Searching for best code with n={n} (num_bits={num_bits}, num_checks={num_checks}) ===")

    for target_d in range(4, 20):
        output_file_name = f"_candidate_code_n{n}_d{target_d}.pkl"

        try:
            result = generator.generate_until_valid(
                output_file_name,
                num_checks,
                num_bits,
                num_checks_per_col,
                code_distance=target_d,
                code_type="regular",
                construction_method="peg"
            )
        except RuntimeError:
            print(f"  d={target_d}: Could not find valid code after max attempts, stopping search for n={n}.")
            break

        # Load the generated matrix and check its actual distance
        with open(output_file_name, "rb") as f:
            parity_check_matrix, *meta, seed = pickle.load(f)

        actual_distance, _, _ = estimate_code_distance(parity_check_matrix)
        print(f"  d={target_d}: actual distance={actual_distance}, seed={seed}")

        if actual_distance > best_distance:
            best_distance = actual_distance
            best_matrix = parity_check_matrix
            best_seed = seed
            best_meta = (num_checks_per_col, "regular", "peg", seed)

    # Save the best matrix found for this n
    if best_matrix is not None:
        best_output = f"_best_code_n{n}.pkl"
        with open(best_output, "wb") as f:
            pickle.dump((best_matrix, *best_meta), f)
        print(f"\n  >> Best code for n={n}: distance={best_distance}, saved to {best_output}")
    else:
        print(f"\n  >> No valid code found for n={n}")


=== Searching for best code with n=4 (num_bits=16, num_checks=12) ===
Attempt 1/200 | Seed: 4128793673
INFO: Regular code: inferred average ones per row as 4
INFO: Before puncturing (if applicable):
INFO: # check nodes = 12
INFO: # variable nodes (bits in codeword) = 16
INFO: # message bits = 4
INFO: # Rate = 0.25

INFO: Puncturing NOT used

Parity check matrix in _parity-check.tmp (sparse format):

 0:  0  4  8 13
 1:  0  5  9 14 15
 2:  1  6  9 12
 3:  2  5 10 13
 4:  3  4 11 14
 5:  2  7 11 15
 6:  2  4 12
 7:  0  6 11
 8:  3  6  8 10 15
 9:  1  7  8 14
10:  1  5 10 12
11:  3  7  9 13

4
[(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (0, 10), (0, 11), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (2, 9), (2, 10), (2, 11), (3, 4), (3, 5), (3, 6), (3, 7), (3, 8), (3, 9), (3, 10), (3, 11), (4, 5), (4, 6), (4, 7), (4, 8), (4, 9), (4, 10), (4, 11), (5, 6), (5, 7), (5, 8), (5, 9)

In [8]:
import pickle
import ldpc.code_util

results = []

for i in range(4, 21):
    filename = f"_best_code_n{i}.pkl"

    try:
        with open(filename, "rb") as file:
            H = pickle.load(file)[0]
    except FileNotFoundError:
        print(f"Skipping missing file: {filename}")
        continue

    n = H.shape[1]
    k = ldpc.code_util.compute_code_dimension(H)

    d_estimate, num_sampled, _ = ldpc.code_util.estimate_code_distance(
        H, timeout_seconds=10
    )

    girth_ok = not ldpc.code_util.search_cycles(H, girth=4)

    print(f"\nFile: {filename}")
    print(f"Code parameters: [n={n}, k={k}, d<={d_estimate}]")
    print(f"Samples: {num_sampled}, Girth > 4: {girth_ok}")

    # store results for later use
    results.append({
        "file": filename,
        "H": H,
        "n": n,
        "k": k,
        "d_est": d_estimate,
        "girth_ok": girth_ok
    })

4
[(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (0, 10), (0, 11), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (2, 9), (2, 10), (2, 11), (3, 4), (3, 5), (3, 6), (3, 7), (3, 8), (3, 9), (3, 10), (3, 11), (4, 5), (4, 6), (4, 7), (4, 8), (4, 9), (4, 10), (4, 11), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (6, 7), (6, 8), (6, 9), (6, 10), (6, 11), (7, 8), (7, 9), (7, 10), (7, 11), (8, 9), (8, 10), (8, 11), (9, 10), (9, 11), (10, 11)]

File: _best_code_n4.pkl
Code parameters: [n=16, k=4, d<=6]
Samples: 41540979, Girth > 4: True
4
[(0, 1), (0, 2), (0, 3), (0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (0, 10), (0, 11), (0, 12), (0, 13), (0, 14), (1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (2, 3), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (2, 9), (2, 10), (2, 11), (2, 12), (2, 13), (2, 14), (3, 4), (3, 5), (3, 6)

In [9]:
for r in results:
    H = r["H"]
    d_estimate = r["d_est"]

    hgp_x, hgp_z = LDPCCodeGenerator.hypergraph_product_code(H, H)

    n_quantum = hgp_x.shape[1]
    k_quantum = hgp_x.shape[1] - hgp_x.shape[0] - hgp_z.shape[0]

    print(f'{r["file"]}: [[n={n_quantum}, k={k_quantum}, d<={d_estimate}]], rate={k_quantum/n_quantum:.4f}')

_best_code_n4.pkl: [[n=400, k=16, d<=6]], rate=0.0400
_best_code_n5.pkl: [[n=625, k=25, d<=8]], rate=0.0400
_best_code_n6.pkl: [[n=900, k=36, d<=8]], rate=0.0400
_best_code_n7.pkl: [[n=1225, k=49, d<=10]], rate=0.0400
_best_code_n8.pkl: [[n=1600, k=64, d<=10]], rate=0.0400
_best_code_n9.pkl: [[n=2025, k=81, d<=10]], rate=0.0400
_best_code_n10.pkl: [[n=2500, k=100, d<=12]], rate=0.0400
_best_code_n11.pkl: [[n=3025, k=121, d<=12]], rate=0.0400
_best_code_n12.pkl: [[n=3600, k=144, d<=12]], rate=0.0400
_best_code_n13.pkl: [[n=4225, k=169, d<=14]], rate=0.0400
_best_code_n14.pkl: [[n=4900, k=196, d<=14]], rate=0.0400
_best_code_n15.pkl: [[n=5625, k=225, d<=14]], rate=0.0400
_best_code_n16.pkl: [[n=6400, k=256, d<=14]], rate=0.0400
_best_code_n17.pkl: [[n=7225, k=289, d<=16]], rate=0.0400
_best_code_n18.pkl: [[n=8100, k=324, d<=16]], rate=0.0400
_best_code_n19.pkl: [[n=9025, k=361, d<=16]], rate=0.0400
_best_code_n20.pkl: [[n=10000, k=400, d<=16]], rate=0.0400


In [12]:
hgp_results = {}

for r in results:
    H = r["H"]
    d_estimate = r["d_est"]

    hgp_x, hgp_z = LDPCCodeGenerator.hypergraph_product_code(H, H)
    n_quantum = hgp_x.shape[1]
    k_quantum = hgp_x.shape[1] - hgp_x.shape[0] - hgp_z.shape[0]
    rate = k_quantum / n_quantum

    code_data = {
        "H":        H,
        "hgp_x":    hgp_x,
        "hgp_z":    hgp_z,
        "n":        n_quantum,
        "k":        k_quantum,
        "d_est":    d_estimate,
        "rate":     rate,
    }

    # Save individual file
    file_name = f"hgp_{n_quantum}_{k_quantum}_{d_estimate}.pkl"
    with open(file_name, "wb") as f:
        pickle.dump(code_data, f)

    hgp_results[file_name] = code_data
    print(f"Saved {file_name}: [[n={n_quantum}, k={k_quantum}, d<={d_estimate}]], rate={rate:.4f}")

print(f"\nSaved {len(hgp_results)} codes total.")

Saved hgp_400_16_6.pkl: [[n=400, k=16, d<=6]], rate=0.0400
Saved hgp_625_25_8.pkl: [[n=625, k=25, d<=8]], rate=0.0400
Saved hgp_900_36_8.pkl: [[n=900, k=36, d<=8]], rate=0.0400
Saved hgp_1225_49_10.pkl: [[n=1225, k=49, d<=10]], rate=0.0400
Saved hgp_1600_64_10.pkl: [[n=1600, k=64, d<=10]], rate=0.0400
Saved hgp_2025_81_10.pkl: [[n=2025, k=81, d<=10]], rate=0.0400
Saved hgp_2500_100_12.pkl: [[n=2500, k=100, d<=12]], rate=0.0400
Saved hgp_3025_121_12.pkl: [[n=3025, k=121, d<=12]], rate=0.0400
Saved hgp_3600_144_12.pkl: [[n=3600, k=144, d<=12]], rate=0.0400
Saved hgp_4225_169_14.pkl: [[n=4225, k=169, d<=14]], rate=0.0400
Saved hgp_4900_196_14.pkl: [[n=4900, k=196, d<=14]], rate=0.0400
Saved hgp_5625_225_14.pkl: [[n=5625, k=225, d<=14]], rate=0.0400
Saved hgp_6400_256_14.pkl: [[n=6400, k=256, d<=14]], rate=0.0400
Saved hgp_7225_289_16.pkl: [[n=7225, k=289, d<=16]], rate=0.0400
Saved hgp_8100_324_16.pkl: [[n=8100, k=324, d<=16]], rate=0.0400
Saved hgp_9025_361_16.pkl: [[n=9025, k=361, d<=16

In [15]:
with open("/Users/aparnagupta/Downloads/notebooks/Measurement_based_distillation/ECC_gen/code_lib/hgp_code_lib/hgp_400_16_6.pkl", "rb") as f:
    code = pickle.load(f)

hgp_x, hgp_z = code["hgp_x"], code["hgp_z"]
print(f"[[n={code['n']}, k={code['k']}, d<={code['d_est']}]]")

[[n=400, k=16, d<=6]]


In [16]:
hgp_x

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 1344 stored elements and shape (192, 400)>